# 01 — Data Creation

Prepares the training catalog from a raw cross-matched table.

**Workflow:**
1. Load the raw catalog
2. Rename columns to the MINAS standard (`mg.FILTERS`, `mg.ERRORS`)
3. Apply quality cuts (you define the criteria)
4. Save the filtered catalog — ready for tuning and training

**Output:** `input_catalog.csv`

## Configuration

Edit the variables in this cell to match your dataset.

In [ ]:
import pandas as pd
import minas as mg

# ── Files ─────────────────────────────────────────────────────────────────────
INPUT_FILE  = 'raw_catalog.csv'   # path to your raw cross-matched catalog
OUTPUT_FILE = 'input_catalog.csv' # output filename used in the next notebooks
INDEX_COL   = 'ID'                # unique identifier column (set to None if not applicable)

# ── Survey ────────────────────────────────────────────────────────────────────
SURVEY = 'SPLUS'   # options: 'SPLUS', 'JPLUS', 'JPAS', 'GAIA', 'WISE'

# ── Column mapping ────────────────────────────────────────────────────────────
# List the column names as they appear in your raw file.
# They will be renamed to the MINAS standard (mg.FILTERS / mg.ERRORS).

COORD_COLS = ['RA', 'DEC']        # coordinate columns → renamed to ['RA', 'DEC']

FILTER_COLS = [                    # magnitude columns → renamed to mg.FILTERS[SURVEY]
    'g_pstotal_0', 'i_pstotal_0', 'j0378_pstotal_0', 'j0395_pstotal_0',
    'j0410_pstotal_0', 'j0430_pstotal_0', 'j0515_pstotal_0',
    'j0660_pstotal_0', 'j0861_pstotal_0', 'r_pstotal_0', 'u_pstotal_0', 'z_pstotal_0',
]

ERROR_COLS = [                     # error columns → renamed to mg.ERRORS[SURVEY]
    'e_g_PStotal', 'e_i_PStotal', 'e_J0378_PStotal', 'e_J0395_PStotal',
    'e_J0410_PStotal', 'e_J0430_PStotal', 'e_J0515_PStotal',
    'e_J0660_PStotal', 'e_J0861_PStotal', 'e_r_PStotal', 'e_u_PStotal', 'e_z_PStotal',
]

PARAM_COLS = ['teff', 'teff_err', 'logg', 'logg_err', 'feh', 'feh_err']  # stellar parameters
OTHER_COLS = ['ID', 'Dist']        # any other columns to keep (e.g. distance, flags)

print(f'Survey  : {SURVEY}')
print(f'Filters : {mg.FILTERS[SURVEY]}')

## Step 1 — Load and rename columns

In [ ]:
# Build rename dictionary
rename_dict  = dict(zip(COORD_COLS,  ['RA', 'DEC']))
rename_dict |= dict(zip(FILTER_COLS, mg.FILTERS[SURVEY]))
rename_dict |= dict(zip(ERROR_COLS,  mg.ERRORS[SURVEY]))

# Load catalog
all_cols = COORD_COLS + FILTER_COLS + ERROR_COLS + PARAM_COLS + OTHER_COLS
df = mg.read_csv(INPUT_FILE, usecols=all_cols)
df = df.rename(columns=rename_dict)

if INDEX_COL and INDEX_COL in df.columns:
    df = df.set_index(INDEX_COL)

print(f'Loaded: {len(df):,} objects, {df.shape[1]} columns')
df.head()

## Step 2 — Apply quality cuts

Edit the filters below according to your science case.
Each `print` shows how many objects remain after each cut.

In [ ]:
n0 = len(df)

# ── Example: remove objects with invalid distance ─────────────────────────────
if 'Dist' in df.columns:
    df = df.dropna(subset=['Dist'])
    df = df[df['Dist'] > 0]
    print(f'After distance filter        : {len(df):,}  (removed {n0 - len(df):,})')

# ── Example: magnitude error cut ──────────────────────────────────────────────
MAX_MAG_ERR = 0.1   # adjust as needed
n = len(df)
for filt in mg.FILTERS[SURVEY]:
    err_col = f'{filt}_err'
    if err_col in df.columns:
        df = df[df[err_col] <= MAX_MAG_ERR]
print(f'After mag error <= {MAX_MAG_ERR}         : {len(df):,}  (removed {n - len(df):,})')

# ── Example: Teff range ───────────────────────────────────────────────────────
teff_col = next((c for c in mg.PARAM_ALIASES['teff'] if c in df.columns), None)
if teff_col:
    n = len(df)
    df = df[(df[teff_col] > 3000) & (df[teff_col] <= 8000)]
    print(f'After Teff in [3000, 8000] K : {len(df):,}  (removed {n - len(df):,})')

# ── Add your own cuts here ────────────────────────────────────────────────────
# df = df[df['some_column'] > threshold]

print(f'\nFinal catalog: {len(df):,} objects  (total removed: {n0 - len(df):,})')

## Step 3 — Save filtered catalog

In [ ]:
df.to_csv(OUTPUT_FILE)
print(f'Catalog saved to: {OUTPUT_FILE}')
print(f'Shape: {df.shape}')